# BLIP+LoRA libre, BLIP+LoRA híbrido, Gemma 3 y Qwen3-VL

Este notebook organiza el flujo completo de entrenamiento, generación, comparación y visualización entre modelos multimodales.

La lógica metodológica usada es:

- **BLIP+LoRA libre base** se entrena con `target_libre_final`.
- **BLIP+LoRA híbrido** se entrena con `target_norm_blip_canonico`.
- `target_norm_blip_canonico` **no es una fuente independiente**: nace de `target_libre_final` mediante normalización/canonización textual.
- **La comparación final de todos los modelos** se realiza frente a `target_libre_final`, renombrado como `descripcion_referencia`.

Flujo de referencia:

```text
target_libre_final
        ↓
normalización / canonización
        ↓
target_norm_blip_canonico
```

Los nombres usados en tablas y figuras son:

- **BLIP+LoRA libre base**
- **BLIP+LoRA híbrido**
- **Gemma 3 4B VLM condicionado**
- **Qwen3-VL fine-tuned**

Además, se generan figuras de apoyo: comparación visual de una célula, heatmap de atributos, matriz de confusión ViT y tabla resumen final.


In [ ]:
# --------------------------------------------------------
# 1. INSTALACIÓN E IMPORTS
# --------------------------------------------------------

!pip install -q transformers accelerate datasets evaluate peft pillow pandas numpy scikit-learn matplotlib tqdm
!pip install -q qwen-vl-utils

import os
import re
import time
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from PIL import Image, ImageFile
from tqdm.auto import tqdm

from sklearn.model_selection import train_test_split

from torch.utils.data import Dataset

from transformers import (
    BlipProcessor,
    BlipForConditionalGeneration,
    TrainingArguments,
    Trainer,
    AutoProcessor,
)

from peft import LoraConfig, get_peft_model, PeftModel

ImageFile.LOAD_TRUNCATED_IMAGES = True


In [ ]:
# --------------------------------------------------------
# 2. DIRECTORIOS Y CONFIGURACIÓN GENERAL
# --------------------------------------------------------

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    pass

if Path("/content/drive/MyDrive/TFM/blood-cell-vllm").exists():
    BASE_DIR = Path("/content/drive/MyDrive/TFM/blood-cell-vllm")
else:
    BASE_DIR = Path(".").resolve()

RUTA_ANALISIS = BASE_DIR / "datos" / "analisis"
RUTA_ANALISIS.mkdir(parents=True, exist_ok=True)

RUTA_DATASET_BASE = RUTA_ANALISIS / "dataset_base_blip_lora.csv"

RUTA_LORA_LIBRE = BASE_DIR / "modelo" / "blip_lora_libre_base"
RUTA_LORA_HIBRIDO = BASE_DIR / "modelo" / "blip_lora_hibrido_canonico"

RUTA_BLIP_LORA_OUT = RUTA_ANALISIS / "salidas_blip_lora_libre_hibrido.csv"
RUTA_GEMMA_OUT = RUTA_ANALISIS / "salidas_gemma3.csv"
RUTA_QWEN_OUT = RUTA_ANALISIS / "salidas_qwen3_vl.csv"

RUTA_COMPARACION_FINAL = RUTA_ANALISIS / "comparacion_blip_lora_gemma3_qwen3_target_libre.csv"

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"

print("BASE_DIR:", BASE_DIR)
print("RUTA_ANALISIS:", RUTA_ANALISIS)
print("Device:", device)


# Rutas de figuras y tablas finales
RUTA_FIGURA_CELULA = RUTA_ANALISIS / "figura_comparativa_modelos_celula.png"
RUTA_HEATMAP_ATRIBUTOS = RUTA_ANALISIS / "heatmap_accuracy_atributos.png"
RUTA_MATRIZ_CONFUSION_VIT = RUTA_ANALISIS / "matriz_confusion_vit.png"
RUTA_TABLA_RESUMEN_FINAL = RUTA_ANALISIS / "tabla_resumen_final_modelos.csv"


## 3. Carga del dataset base

El dataset debe contener, como mínimo, las siguientes columnas:

- `imagen`
- `clase_real`
- `instruction_blip`
- `target_libre_final`
- `target_norm_blip_canonico`

Si el target canónico no existiera, se puede crear a partir de una función de canonización aplicada sobre una versión normalizada previa. En este notebook se comprueba su existencia para evitar entrenar accidentalmente con una columna equivocada.


In [ ]:
# --------------------------------------------------------
# 3. CARGA DEL DATASET BASE
# --------------------------------------------------------

df = pd.read_csv(RUTA_DATASET_BASE).fillna("")

COLUMNAS_NECESARIAS = [
    "imagen",
    "clase_real",
    "instruction_blip",
    "target_libre_final",
    "target_norm_blip_canonico",
]

faltantes = [c for c in COLUMNAS_NECESARIAS if c not in df.columns]
if faltantes:
    raise ValueError(f"Faltan columnas necesarias en el dataset: {faltantes}")

df = df[df["imagen"].astype(str).str.strip() != ""].copy()
df = df[df["target_libre_final"].astype(str).str.strip() != ""].copy()
df = df[df["target_norm_blip_canonico"].astype(str).str.strip() != ""].copy()
df = df.reset_index(drop=True)

print("Dataset cargado:", df.shape)
display(df.head())


In [ ]:
# --------------------------------------------------------
# 3B. COMPROBACIÓN DE TARGETS
# --------------------------------------------------------

COLUMNAS_REQUERIDAS = [
    "imagen",
    "clase_real",
    "instruction_blip",
    "target_libre_final",
    "target_norm_blip_canonico",
]

faltan = [c for c in COLUMNAS_REQUERIDAS if c not in df.columns]
if faltan:
    raise ValueError(f"Faltan columnas necesarias en el dataset: {faltan}")

print("Targets comprobados correctamente:")
print("- BLIP+LoRA libre base -> target_libre_final")
print("- BLIP+LoRA híbrido -> target_norm_blip_canonico")
print("- Evaluación final común -> target_libre_final")
print("Nota: target_norm_blip_canonico deriva de target_libre_final.")


In [ ]:
# --------------------------------------------------------
# 4. SPLIT TRAIN / VAL
# --------------------------------------------------------

train_df, val_df = train_test_split(
    df,
    test_size=0.15,
    random_state=SEED,
    stratify=df["clase_real"] if "clase_real" in df.columns else None,
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print("Train:", len(train_df))
print("Val:", len(val_df))


In [ ]:
# --------------------------------------------------------
# 5. CONFIGURACIÓN BLIP + LoRA
# --------------------------------------------------------

PROCESSOR_NAME = "Salesforce/blip-image-captioning-base"
MODEL_NAME = "Salesforce/blip-image-captioning-base"

COLUMNA_IMAGEN = "imagen"
COLUMNA_INPUT = "instruction_blip"

MAX_LENGTH_TEXT = 128

LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05

BATCH_SIZE = 4
EPOCHS = 3
LEARNING_RATE = 2e-5


In [ ]:
# --------------------------------------------------------
# 6. DATASET REUTILIZABLE PARA BLIP+LoRA
# --------------------------------------------------------

class BlipLoraDataset(Dataset):
    def __init__(self, df, processor, columna_target):
        self.df = df.reset_index(drop=True)
        self.processor = processor
        self.tokenizer = processor.tokenizer
        self.image_processor = processor.image_processor
        self.pad_token_id = self.tokenizer.pad_token_id
        self.columna_target = columna_target

        if self.columna_target not in self.df.columns:
            raise ValueError(f"No existe la columna target: {self.columna_target}")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image_path = str(row[COLUMNA_IMAGEN]).strip()
        prompt = str(row[COLUMNA_INPUT]).strip()
        target = str(row[self.columna_target]).strip()

        image = Image.open(image_path).convert("RGB")
        full_text = f"{prompt} {target}".strip()

        image_inputs = self.image_processor(images=image)
        pixel_values = torch.tensor(
            image_inputs["pixel_values"][0],
            dtype=torch.float32,
        )

        text_inputs = self.tokenizer(
            full_text,
            padding="max_length",
            truncation=True,
            max_length=MAX_LENGTH_TEXT,
            return_tensors="pt",
        )

        prompt_inputs = self.tokenizer(
            prompt,
            padding=False,
            truncation=True,
            max_length=MAX_LENGTH_TEXT,
            return_tensors="pt",
        )

        input_ids = text_inputs["input_ids"].squeeze(0)
        attention_mask = text_inputs["attention_mask"].squeeze(0)

        labels = input_ids.clone()
        labels[labels == self.pad_token_id] = -100

        prompt_len = prompt_inputs["input_ids"].shape[1]
        prompt_len = min(prompt_len, labels.shape[0])
        labels[:prompt_len] = -100

        return {
            "pixel_values": pixel_values,
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
        }


In [ ]:
# --------------------------------------------------------
# 7. FUNCIONES PARA PREPARAR BLIP+LoRA
# --------------------------------------------------------

def detectar_modulos_lora_blip(model):
    modulos = []
    for name, _ in model.named_modules():
        if "text_decoder.bert.encoder.layer" not in name:
            continue
        if (
            name.endswith("attention.self.query")
            or name.endswith("attention.self.value")
            or name.endswith("crossattention.self.query")
            or name.endswith("crossattention.self.value")
        ):
            modulos.append(name)
    return sorted(set(modulos))


def preparar_modelo_blip_lora():
    processor = BlipProcessor.from_pretrained(PROCESSOR_NAME)
    model = BlipForConditionalGeneration.from_pretrained(MODEL_NAME)
    model.config.tie_word_embeddings = False

    target_modules = detectar_modulos_lora_blip(model)
    print("Módulos LoRA detectados:", len(target_modules))

    if len(target_modules) == 0:
        raise ValueError("No se detectaron módulos compatibles con LoRA en BLIP.")

    lora_config = LoraConfig(
        inference_mode=False,
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules=target_modules,
    )

    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

    return processor, model


## 8. Entrenamiento BLIP+LoRA libre base

Este modelo usa como objetivo:

```python
target_libre_final
```

Es decir, la descripción libre final generada en la etapa previa.


In [ ]:
# --------------------------------------------------------
# 8. ENTRENAMIENTO BLIP+LoRA LIBRE BASE
# --------------------------------------------------------

COLUMNA_TARGET_LIBRE = "target_libre_final"
print("Target entrenamiento libre:", COLUMNA_TARGET_LIBRE)

processor_libre, model_libre = preparar_modelo_blip_lora()

train_dataset_libre = BlipLoraDataset(train_df, processor_libre, COLUMNA_TARGET_LIBRE)
val_dataset_libre = BlipLoraDataset(val_df, processor_libre, COLUMNA_TARGET_LIBRE)

training_args_libre = TrainingArguments(
    output_dir=str(RUTA_LORA_LIBRE),
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    remove_unused_columns=False,
    report_to="none",
    seed=SEED,
)

trainer_libre = Trainer(
    model=model_libre,
    args=training_args_libre,
    train_dataset=train_dataset_libre,
    eval_dataset=val_dataset_libre,
)

trainer_libre.train()
trainer_libre.save_model(RUTA_LORA_LIBRE)
processor_libre.save_pretrained(RUTA_LORA_LIBRE)

print("BLIP+LoRA libre base guardado en:", RUTA_LORA_LIBRE)


## 9. Entrenamiento BLIP+LoRA híbrido

Este modelo usa como objetivo:

```python
target_norm_blip_canonico
```

Este target **nace de `target_libre_final`**. No se considera una fuente externa distinta, sino una versión normalizada y canonizada del target libre.

Flujo:

```text
target_libre_final
        ↓
target_norm
        ↓
target_norm_blip_canonico
```

Por eso el entrenamiento híbrido/canónico utiliza `target_norm_blip_canonico`, mientras que la comparación final se realiza contra `target_libre_final`.


In [ ]:
# --------------------------------------------------------
# 9. ENTRENAMIENTO BLIP+LoRA HÍBRIDO
# --------------------------------------------------------

COLUMNA_TARGET_HIBRIDO = "target_norm_blip_canonico"
print("Target entrenamiento híbrido/canónico:", COLUMNA_TARGET_HIBRIDO, "(derivado de target_libre_final)")

processor_hibrido, model_hibrido = preparar_modelo_blip_lora()

train_dataset_hibrido = BlipLoraDataset(train_df, processor_hibrido, COLUMNA_TARGET_HIBRIDO)
val_dataset_hibrido = BlipLoraDataset(val_df, processor_hibrido, COLUMNA_TARGET_HIBRIDO)

training_args_hibrido = TrainingArguments(
    output_dir=str(RUTA_LORA_HIBRIDO),
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    remove_unused_columns=False,
    report_to="none",
    seed=SEED,
)

trainer_hibrido = Trainer(
    model=model_hibrido,
    args=training_args_hibrido,
    train_dataset=train_dataset_hibrido,
    eval_dataset=val_dataset_hibrido,
)

trainer_hibrido.train()
trainer_hibrido.save_model(RUTA_LORA_HIBRIDO)
processor_hibrido.save_pretrained(RUTA_LORA_HIBRIDO)

print("BLIP+LoRA híbrido/canónico guardado en:", RUTA_LORA_HIBRIDO)


In [ ]:
# --------------------------------------------------------
# 10. FUNCIONES DE INFERENCIA BLIP+LoRA
# --------------------------------------------------------

def limpiar_salida_modelo(texto):
    texto = str(texto).strip()
    texto = re.sub(r"\s+", " ", texto)
    texto = texto.replace(" .", ".")
    return texto


def cargar_blip_lora(ruta_lora):
    processor = BlipProcessor.from_pretrained(ruta_lora)
    base_model = BlipForConditionalGeneration.from_pretrained(MODEL_NAME)
    base_model.config.tie_word_embeddings = False
    model = PeftModel.from_pretrained(base_model, ruta_lora)
    model.to(device)
    model.eval()
    return processor, model


def generar_blip_lora(image_path, prompt, processor, model, max_new_tokens=80):
    image = Image.open(image_path).convert("RGB")

    inputs = processor(
        images=image,
        text=prompt,
        return_tensors="pt",
    ).to(device)

    with torch.inference_mode():
        ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=4,
            do_sample=False,
            repetition_penalty=1.1,
        )

    raw = processor.decode(ids[0], skip_special_tokens=True)
    final = limpiar_salida_modelo(raw)

    if final.lower().startswith(prompt.lower()):
        final = final[len(prompt):].strip()

    return limpiar_salida_modelo(final)


In [ ]:
# --------------------------------------------------------
# 11. GENERACIÓN BLIP+LoRA LIBRE E HÍBRIDO
# --------------------------------------------------------

PROMPT_BLIP_LORA_BASE = "Describe the cell morphology."

def construir_prompt_hibrido(row):
    return f"""
Describe the morphology of the cell using the visual information and the provided class context.

Cell class: {row['clase_real']}

Return one concise morphology sentence.
""".strip()


processor_libre_inf, model_libre_inf = cargar_blip_lora(RUTA_LORA_LIBRE)
processor_hibrido_inf, model_hibrido_inf = cargar_blip_lora(RUTA_LORA_HIBRIDO)

if RUTA_BLIP_LORA_OUT.exists():
    df_blip_out = pd.read_csv(RUTA_BLIP_LORA_OUT).fillna("")
    if len(df_blip_out) != len(df):
        df_blip_out = df.copy()
else:
    df_blip_out = df.copy()

for col in [
    "descripcion_generada_blip_lora_base",
    "descripcion_generada_blip_lora_hibrido",
    "error_blip_lora_base",
    "error_blip_lora_hibrido",
]:
    if col not in df_blip_out.columns:
        df_blip_out[col] = ""

for idx in tqdm(range(len(df_blip_out)), desc="Generando BLIP+LoRA libre/híbrido"):
    image_path = str(df_blip_out.at[idx, "imagen"]).strip()

    if not str(df_blip_out.at[idx, "descripcion_generada_blip_lora_base"]).strip():
        try:
            df_blip_out.at[idx, "descripcion_generada_blip_lora_base"] = generar_blip_lora(
                image_path,
                PROMPT_BLIP_LORA_BASE,
                processor_libre_inf,
                model_libre_inf,
            )
            df_blip_out.at[idx, "error_blip_lora_base"] = ""
        except Exception as exc:
            df_blip_out.at[idx, "error_blip_lora_base"] = f"{type(exc).__name__}: {exc}"

    if not str(df_blip_out.at[idx, "descripcion_generada_blip_lora_hibrido"]).strip():
        try:
            prompt_hibrido = construir_prompt_hibrido(df_blip_out.iloc[idx])
            df_blip_out.at[idx, "descripcion_generada_blip_lora_hibrido"] = generar_blip_lora(
                image_path,
                prompt_hibrido,
                processor_hibrido_inf,
                model_hibrido_inf,
            )
            df_blip_out.at[idx, "error_blip_lora_hibrido"] = ""
        except Exception as exc:
            df_blip_out.at[idx, "error_blip_lora_hibrido"] = f"{type(exc).__name__}: {exc}"

    if (idx + 1) % 25 == 0:
        df_blip_out.to_csv(RUTA_BLIP_LORA_OUT, index=False, encoding="utf-8")

df_blip_out.to_csv(RUTA_BLIP_LORA_OUT, index=False, encoding="utf-8")

print("Salidas BLIP+LoRA guardadas en:", RUTA_BLIP_LORA_OUT)
display(df_blip_out.head())


## 12. Gemma 3

Este bloque asume que existe un modelo Gemma 3 VLM disponible en el entorno. Puede requerir ajustar el nombre exacto del modelo según la versión usada.


In [ ]:
# --------------------------------------------------------
# 12. GEMMA 3
# --------------------------------------------------------
# Ajusta MODEL_GEMMA al identificador exacto que estés usando.

MODEL_GEMMA = "google/gemma-3-4b-it"
MAX_NEW_TOKENS_GEMMA = 180
GUARDAR_CADA = 25

def construir_prompt_gemma(row):
    return f"""
You are a cell morphology captioning assistant.

Cell class: {row['clase_real']}
Reference context: {row['target_libre_final']}
Initial BLIP+LoRA hybrid caption: {row.get('descripcion_generada_blip_lora_hibrido', '')}

Task:
Generate one precise morphology sentence.
Use the image as the primary evidence.
Do not add unsupported findings.
Return only the final caption.
""".strip()

# Este bloque puede necesitar adaptación según el modelo Gemma 3 VLM exacto.
# Si ya tienes un bloque funcional de Gemma, conserva tu carga original y usa:
# - construir_prompt_gemma(row)
# - columna de salida: prediccion_gemma3

print("Bloque Gemma 3 preparado. Inserta aquí tu carga exacta si ya la tienes funcionando.")


## 13. Qwen3-VL

Este bloque está preparado para Qwen3-VL. Si el entorno de Hugging Face cambia el nombre exacto de la clase/modelo, ajusta únicamente `MODEL_QWEN` y la clase de carga.


In [ ]:
# --------------------------------------------------------
# 13. QWEN3-VL
# --------------------------------------------------------

# Nota: ajusta el identificador si usas otra variante exacta.
MODEL_QWEN = "Qwen/Qwen3-VL-4B-Instruct"
MAX_NEW_TOKENS_QWEN = 180
GUARDAR_CADA = 25
ESPERA_SEGUNDOS = 0.5

try:
    from transformers import AutoProcessor, AutoModelForImageTextToText
    from qwen_vl_utils import process_vision_info
except Exception as exc:
    print("Faltan dependencias para Qwen3-VL:", exc)


def construir_prompt_qwen(row):
    return f"""
You are a cell morphology captioning assistant.

Cell class: {row['clase_real']}
Reference context: {row['target_libre_final']}
Initial BLIP+LoRA hybrid caption: {row.get('descripcion_generada_blip_lora_hibrido', '')}

Task:
Generate one precise morphology sentence.
Use the image as the primary evidence.
Do not add unsupported findings.
Return only the final caption.
""".strip()


# Si tu entorno ya carga Qwen3-VL con una clase concreta, sustituye solo estas dos líneas.
# processor_qwen = AutoProcessor.from_pretrained(MODEL_QWEN, trust_remote_code=True)
# model_qwen = AutoModelForImageTextToText.from_pretrained(
#     MODEL_QWEN,
#     torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
#     device_map="auto",
#     trust_remote_code=True,
# )
# model_qwen.eval()

print("Bloque Qwen3-VL preparado. Activa la carga cuando tengas el modelo disponible en el entorno.")


## 14. Tabla final de comparación

Aunque el modelo híbrido se entrena con `target_norm_blip_canonico`, esa columna nace de `target_libre_final`.

Para comparar todos los modelos con una referencia común, se usa:

```python
target_libre_final -> descripcion_referencia
```

Así, BLIP+LoRA libre, BLIP+LoRA híbrido, Gemma 3 y Qwen3-VL se evalúan frente al mismo texto de referencia.


In [ ]:
# --------------------------------------------------------
# 14. TABLA FINAL DE COMPARACIÓN DE SALIDAS
# --------------------------------------------------------

df_comp = pd.read_csv(RUTA_BLIP_LORA_OUT).fillna("")

comparacion = df_comp[[
    "imagen",
    "clase_real",
    "target_libre_final",
    "target_norm_blip_canonico",
    "descripcion_generada_blip_lora_base",
    "descripcion_generada_blip_lora_hibrido",
]].copy()

comparacion = comparacion.rename(columns={
    "target_libre_final": "descripcion_referencia"
})

if RUTA_GEMMA_OUT.exists():
    gemma_tmp = pd.read_csv(RUTA_GEMMA_OUT).fillna("")
    if "prediccion_gemma3" in gemma_tmp.columns:
        comparacion["prediccion_gemma3"] = gemma_tmp["prediccion_gemma3"].astype(str)

if RUTA_QWEN_OUT.exists():
    qwen_tmp = pd.read_csv(RUTA_QWEN_OUT).fillna("")
    if "prediccion_qwen_vl" in qwen_tmp.columns:
        comparacion["prediccion_qwen_vl"] = qwen_tmp["prediccion_qwen_vl"].astype(str)

comparacion.to_csv(RUTA_COMPARACION_FINAL, index=False, encoding="utf-8")

print("Comparación final guardada en:", RUTA_COMPARACION_FINAL)
display(comparacion.head())


In [ ]:
# --------------------------------------------------------
# 15. MÉTRICAS DE COMPARACIÓN ENTRE MODELOS
# --------------------------------------------------------

RUTA_METRICAS_OUT = RUTA_ANALISIS / "metricas_comparacion_modelos_target_libre.csv"
RUTA_HEATMAP_OUT = RUTA_ANALISIS / "heatmap_metricas_modelos_target_libre.png"
RUTA_RANKING_OUT = RUTA_ANALISIS / "ranking_modelos_target_libre.csv"
RUTA_DETALLE_METRICAS_OUT = RUTA_ANALISIS / "detalle_metricas_modelos_target_libre.csv"

df_eval = comparacion.copy().fillna("")

modelos = {
    "BLIP+LoRA libre base (%)": "descripcion_generada_blip_lora_base",
    "BLIP+LoRA híbrido (%)": "descripcion_generada_blip_lora_hibrido",
    "Gemma 3 4B VLM condicionado (%)": "prediccion_gemma3",
    "Qwen3-VL fine-tuned (%)": "prediccion_qwen_vl",
}

modelos = {k: v for k, v in modelos.items() if v in df_eval.columns}

print("Referencia de evaluación: descripcion_referencia = target_libre_final")
print("Modelos evaluados:")
for nombre, col in modelos.items():
    print(f"- {nombre}: {col}")


def limpiar_texto_eval(texto):
    texto = str(texto).lower().strip()
    texto = re.sub(r"\s+", " ", texto)
    return texto


def tokenizar(texto):
    return set(re.findall(r"[a-zA-ZáéíóúñüÁÉÍÓÚÑÜ]+", limpiar_texto_eval(texto)))


def calcular_similitud(referencia, prediccion):
    ref_tokens = tokenizar(referencia)
    pred_tokens = tokenizar(prediccion)

    if len(ref_tokens) == 0:
        return 0.0

    return len(ref_tokens & pred_tokens) / len(ref_tokens)


def clasificar_similitud(score, prediccion):
    if not str(prediccion).strip():
        return "sin salida"
    elif score >= 0.70:
        return "alta similitud"
    elif score >= 0.40:
        return "similitud parcial"
    else:
        return "baja similitud"


resultados = []
detalle_metricas = df_eval[["imagen", "clase_real", "descripcion_referencia"]].copy()

for nombre_modelo, col_pred in modelos.items():
    scores = []
    categorias = []

    for _, row in df_eval.iterrows():
        score = calcular_similitud(
            row["descripcion_referencia"],
            row[col_pred],
        )

        categoria = clasificar_similitud(score, row[col_pred])

        scores.append(score)
        categorias.append(categoria)

    total = len(df_eval)

    alta = sum(c == "alta similitud" for c in categorias)
    parcial = sum(c == "similitud parcial" for c in categorias)
    baja = sum(c == "baja similitud" for c in categorias)
    sin_salida = sum(c == "sin salida" for c in categorias)

    resultados.append({
        "Modelo": nombre_modelo,
        "n": total,
        "% alta similitud": round(alta / total * 100, 2),
        "% similitud aceptable": round((alta + parcial) / total * 100, 2),
        "% similitud parcial": round(parcial / total * 100, 2),
        "% baja similitud": round(baja / total * 100, 2),
        "% sin salida": round(sin_salida / total * 100, 2),
        "similitud media": round(np.mean(scores) * 100, 2),
    })

    detalle_metricas[f"score_{nombre_modelo}"] = scores
    detalle_metricas[f"categoria_{nombre_modelo}"] = categorias

tabla_metricas = pd.DataFrame(resultados)
tabla_metricas.to_csv(RUTA_METRICAS_OUT, index=False, encoding="utf-8")
detalle_metricas.to_csv(RUTA_DETALLE_METRICAS_OUT, index=False, encoding="utf-8")

display(tabla_metricas)
print("Métricas guardadas en:", RUTA_METRICAS_OUT)
print("Detalle por imagen guardado en:", RUTA_DETALLE_METRICAS_OUT)


In [ ]:
# --------------------------------------------------------
# 16. HEATMAP DE MÉTRICAS GLOBALES
# --------------------------------------------------------

heatmap_df = tabla_metricas.set_index("Modelo")[[
    "% alta similitud",
    "% similitud aceptable",
    "% similitud parcial",
    "% baja similitud",
    "similitud media",
]]

fig, ax = plt.subplots(figsize=(13, 5))

im = ax.imshow(heatmap_df, aspect="auto", vmin=0, vmax=100)

ax.set_xticks(range(len(heatmap_df.columns)))
ax.set_xticklabels(heatmap_df.columns, rotation=30, ha="right")

ax.set_yticks(range(len(heatmap_df.index)))
ax.set_yticklabels(heatmap_df.index)

for i in range(len(heatmap_df.index)):
    for j in range(len(heatmap_df.columns)):
        ax.text(
            j,
            i,
            f"{heatmap_df.iloc[i, j]:.1f}",
            ha="center",
            va="center",
            fontweight="bold",
        )

cbar = fig.colorbar(im, ax=ax)
cbar.set_label("Porcentaje (%)")

ax.set_title("Comparación agregada entre modelos frente a target_libre_final")
plt.tight_layout()
plt.savefig(RUTA_HEATMAP_OUT, dpi=300, bbox_inches="tight")
plt.show()

print("Heatmap guardado en:", RUTA_HEATMAP_OUT)


In [ ]:
# --------------------------------------------------------
# 17. RANKING DE MODELOS SEGÚN RENDIMIENTO GLOBAL
# --------------------------------------------------------

ranking_modelos = tabla_metricas.sort_values(
    by=["% similitud aceptable", "similitud media"],
    ascending=False,
).reset_index(drop=True)

ranking_modelos.to_csv(RUTA_RANKING_OUT, index=False, encoding="utf-8")

display(ranking_modelos)
print("Ranking guardado en:", RUTA_RANKING_OUT)


## 18. Figuras y tablas finales

Esta sección añade las figuras de apoyo usadas para interpretar la comparación:

- Figura comparativa de una célula seleccionada.
- Heatmap de accuracy por atributo morfológico.
- Matriz de confusión ViT normalizada por clase real.
- Tabla resumen final con métricas agregadas.


In [ ]:
# --------------------------------------------------------
# 18A. FIGURA COMPARATIVA DE UNA CÉLULA SELECCIONADA
# --------------------------------------------------------

RUTA_FIGURA_CELULA = RUTA_ANALISIS / "figura_comparativa_modelos_celula.png"

# Selección: si hay clase band, se prioriza; si no, se toma la primera fila válida.
df_fig = comparacion.copy().fillna("")
if "band" in set(df_fig["clase_real"].astype(str).str.lower()):
    fila = df_fig[df_fig["clase_real"].astype(str).str.lower() == "band"].iloc[0]
else:
    fila = df_fig.iloc[0]


def resumir_descripcion(texto, max_chars=95):
    texto = " ".join(str(texto).strip().split())
    if len(texto) <= max_chars:
        return texto
    return texto[:max_chars].rstrip() + "..."


def comentario_por_score(ref, pred):
    score = calcular_similitud(ref, pred)
    if score >= 0.70:
        return "Alta concordancia"
    elif score >= 0.40:
        return "Concordancia parcial"
    elif str(pred).strip():
        return "Baja concordancia"
    return "Sin salida"

ref = fila["descripcion_referencia"]
filas_tabla = [["Target experto", resumir_descripcion(ref), "Referencia"]]

modelos_figura = [
    ("BLIP+LoRA híbrido", "descripcion_generada_blip_lora_hibrido"),
    ("Gemma 3 4B", "prediccion_gemma3"),
    ("Qwen3-VL", "prediccion_qwen_vl"),
]

for nombre, columna in modelos_figura:
    if columna in df_fig.columns:
        pred = fila[columna]
        filas_tabla.append([
            nombre,
            resumir_descripcion(pred),
            comentario_por_score(ref, pred),
        ])

fig = plt.figure(figsize=(13, 10))
gs = fig.add_gridspec(2, 1, height_ratios=[3, 2.2])

ax_img = fig.add_subplot(gs[0])
ax_img.axis("off")

try:
    img = Image.open(str(fila["imagen"])).convert("RGB")
    ax_img.imshow(img)
except Exception as exc:
    ax_img.text(0.5, 0.5, f"No se pudo cargar la imagen\n{exc}", ha="center", va="center")

ax_img.set_title(
    f"Imagen celular seleccionada — clase real: {fila['clase_real']}",
    fontsize=20,
    fontweight="bold",
    pad=18,
)

ax_tabla = fig.add_subplot(gs[1])
ax_tabla.axis("off")

tabla = ax_tabla.table(
    cellText=filas_tabla,
    colLabels=["Modelo", "Descripción breve", "Comentario"],
    loc="center",
    cellLoc="left",
    colLoc="left",
    colWidths=[0.20, 0.65, 0.15],
)

tabla.auto_set_font_size(False)
tabla.set_fontsize(13)
tabla.scale(1, 2.1)

for (row, col), cell in tabla.get_celld().items():
    cell.set_linewidth(0.8)
    if row == 0:
        cell.set_facecolor("#d9e2ef")
        cell.set_text_props(weight="bold")
    if col == 0 and row > 0:
        cell.set_text_props(weight="bold")

plt.tight_layout()
plt.savefig(RUTA_FIGURA_CELULA, dpi=300, bbox_inches="tight")
plt.show()

print("Figura comparativa guardada en:", RUTA_FIGURA_CELULA)


In [ ]:
# --------------------------------------------------------
# 18B. HEATMAP DE ACCURACY POR ATRIBUTO
# --------------------------------------------------------

RUTA_HEATMAP_ATRIBUTOS = RUTA_ANALISIS / "heatmap_accuracy_atributos.png"
RUTA_TABLA_ATRIBUTOS = RUTA_ANALISIS / "tabla_accuracy_atributos.csv"

# Diccionario deliberadamente general: evalúa componentes morfológicos principales
# sin exponer una lista excesivamente específica de reglas internas.

ATRIBUTOS = {
    "clase celular": [],
    "tamano celular": [],
    "forma nuclear": [],
    "cromatina": [],
    "citoplasma": [],
    "granulacion": [],
}

def contiene_termino(texto, terminos):
    texto = limpiar_texto_eval(texto)
    return any(t.lower() in texto for t in terminos)


resultados_atributos = []

for atributo, terminos in ATRIBUTOS.items():
    fila_resultado = {"atributo": atributo}
    ref_presente = df_eval["descripcion_referencia"].apply(lambda x: contiene_termino(x, terminos))

    for nombre_modelo, col_pred in modelos.items():
        pred_presente = df_eval[col_pred].apply(lambda x: contiene_termino(x, terminos))
        accuracy_atributo = (pred_presente == ref_presente).mean() * 100
        fila_resultado[nombre_modelo] = round(float(accuracy_atributo), 1)

    resultados_atributos.append(fila_resultado)

tabla_atributos = pd.DataFrame(resultados_atributos).set_index("atributo")
tabla_atributos.to_csv(RUTA_TABLA_ATRIBUTOS, encoding="utf-8")

display(tabla_atributos)

fig, ax = plt.subplots(figsize=(13, 7))
im = ax.imshow(tabla_atributos, aspect="auto", vmin=0, vmax=100)

ax.set_xticks(range(len(tabla_atributos.columns)))
ax.set_xticklabels(tabla_atributos.columns, rotation=30, ha="right")

ax.set_yticks(range(len(tabla_atributos.index)))
ax.set_yticklabels(tabla_atributos.index)

for i in range(len(tabla_atributos.index)):
    for j in range(len(tabla_atributos.columns)):
        ax.text(
            j,
            i,
            f"{tabla_atributos.iloc[i, j]:.1f}",
            ha="center",
            va="center",
            fontweight="bold",
        )

cbar = fig.colorbar(im, ax=ax)
cbar.set_label("Accuracy (%)")

ax.set_title(
    "Heatmap de accuracy por atributo\n"
    "BLIP+LoRA libre base vs BLIP+LoRA híbrido vs Gemma 3 4B VLM condicionado vs Qwen3-VL",
    fontsize=16,
)

plt.tight_layout()
plt.savefig(RUTA_HEATMAP_ATRIBUTOS, dpi=300, bbox_inches="tight")
plt.show()

print("Heatmap de atributos guardado en:", RUTA_HEATMAP_ATRIBUTOS)
print("Tabla de atributos guardada en:", RUTA_TABLA_ATRIBUTOS)


In [ ]:
# --------------------------------------------------------
# 18C. MATRIZ DE CONFUSIÓN ViT NORMALIZADA POR CLASE REAL
# --------------------------------------------------------

from sklearn.metrics import confusion_matrix

RUTA_MATRIZ_CONFUSION_VIT = RUTA_ANALISIS / "matriz_confusion_vit.png"

# Se busca una tabla que contenga clase_real y clase_predicha.
# Si tu notebook ya genera df_final o df_clean con estas columnas, se usará directamente.
if "df_clean" in globals() and {"clase_real", "clase_predicha"}.issubset(df_clean.columns):
    df_vit_conf = df_clean.copy()
elif "df_final" in globals() and {"clase_real", "clase_predicha"}.issubset(df_final.columns):
    df_vit_conf = df_final.copy()
else:
    rutas_posibles_vit = [
        BASE_DIR / "datos" / "dataset_final_VIT_BLIP_patologo_Diana.csv",
        BASE_DIR / "datos" / "dataset_base_clean_patologo_Diana.csv",
        RUTA_ANALISIS / "dataset_final_VIT_BLIP_patologo_Diana.csv",
    ]
    df_vit_conf = None
    for ruta in rutas_posibles_vit:
        if ruta.exists():
            tmp = pd.read_csv(ruta).fillna("")
            if {"clase_real", "clase_predicha"}.issubset(tmp.columns):
                df_vit_conf = tmp.copy()
                print("Datos ViT cargados desde:", ruta)
                break

if df_vit_conf is None:
    print("No se encontró una tabla con columnas clase_real y clase_predicha. Se omite la matriz de confusión ViT.")
else:
    df_vit_conf["clase_real"] = df_vit_conf["clase_real"].astype(str).str.strip().str.lower()
    df_vit_conf["clase_predicha"] = df_vit_conf["clase_predicha"].astype(str).str.strip().str.lower()
    df_vit_conf = df_vit_conf[df_vit_conf["clase_predicha"] != "error"].copy()

    clases = [
        "basophil", "eosinophil", "erythroblast", "lymphocyte", "monocyte", "platelet",
        "promyelocyte", "myelocyte", "metamyelocyte", "ig", "band", "segmented",
    ]
    clases = [c for c in clases if c in set(df_vit_conf["clase_real"]) or c in set(df_vit_conf["clase_predicha"])]

    cm = confusion_matrix(
        df_vit_conf["clase_real"],
        df_vit_conf["clase_predicha"],
        labels=clases,
    )

    cm_norm = cm.astype(float) / np.maximum(cm.sum(axis=1, keepdims=True), 1) * 100

    fig, ax = plt.subplots(figsize=(12, 10))
    im = ax.imshow(cm_norm, vmin=0, vmax=100)

    ax.set_xticks(range(len(clases)))
    ax.set_xticklabels(clases, rotation=45, ha="right")
    ax.set_yticks(range(len(clases)))
    ax.set_yticklabels(clases)

    for i in range(len(clases)):
        for j in range(len(clases)):
            val = cm_norm[i, j]
            ax.text(
                j,
                i,
                f"{val:.1f}",
                ha="center",
                va="center",
                color="white" if val >= 70 else "black",
                fontsize=9,
            )

    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label("%")

    ax.set_title("Matriz de confusión normalizada por clase real", fontsize=16)
    ax.set_xlabel("Clase predicha")
    ax.set_ylabel("Clase real")
    fig.text(0.5, 0.02, "Los valores representan porcentajes normalizados por fila.", ha="center")

    plt.tight_layout(rect=[0, 0.04, 1, 1])
    plt.savefig(RUTA_MATRIZ_CONFUSION_VIT, dpi=300, bbox_inches="tight")
    plt.show()

    print("Matriz de confusión guardada en:", RUTA_MATRIZ_CONFUSION_VIT)


In [ ]:
# --------------------------------------------------------
# 18D. TABLA RESUMEN FINAL DE RESULTADOS
# --------------------------------------------------------

RUTA_TABLA_RESUMEN_FINAL = RUTA_ANALISIS / "tabla_resumen_final_modelos.csv"

# Accuracy media por atributos, si se ha calculado el heatmap anterior.
if "tabla_atributos" in globals():
    acc_atributos_media = tabla_atributos.mean(axis=0).to_dict()
else:
    acc_atributos_media = {}

# Mejora respecto al input/base: se calcula respecto a BLIP+LoRA libre base cuando existe.
col_base_score = "score_BLIP+LoRA libre base (%)"

def calcular_mejora_respecto_base(nombre_modelo):
    col_modelo = f"score_{nombre_modelo}"
    if nombre_modelo == "BLIP+LoRA libre base (%)":
        return "—"
    if col_base_score not in detalle_metricas.columns or col_modelo not in detalle_metricas.columns:
        return "N/D"
    mejora = (detalle_metricas[col_modelo] > detalle_metricas[col_base_score]).mean() * 100
    return round(float(mejora), 1)

filas_resumen = []
for _, row in tabla_metricas.iterrows():
    modelo = row["Modelo"]
    filas_resumen.append({
        "Modelo": modelo,
        "n": int(row["n"]),
        "Corr. estricto": row["% alta similitud"],
        "Error leve de atributos": row["% similitud parcial"],
        "Corr. ampliado": row["% similitud aceptable"],
        "Alta adecuación": row["% alta similitud"],
        "Mejora respecto al input": calcular_mejora_respecto_base(modelo),
        "Acc. atributos": round(float(acc_atributos_media.get(modelo, row["similitud media"])), 1),
    })

tabla_resumen_final = pd.DataFrame(filas_resumen)
tabla_resumen_final.to_csv(RUTA_TABLA_RESUMEN_FINAL, index=False, encoding="utf-8")

display(tabla_resumen_final)
print("Tabla resumen final guardada en:", RUTA_TABLA_RESUMEN_FINAL)


In [ ]:
# --------------------------------------------------------
# 19. RESUMEN AUTOMÁTICO DE RESULTADOS
# --------------------------------------------------------

mejor_global = ranking_modelos.iloc[0]

print(f"""
Resumen:

Se compararon las descripciones generadas por BLIP+LoRA libre base,
BLIP+LoRA híbrido, Gemma 3 4B VLM condicionado y Qwen3-VL fine-tuned
utilizando como referencia común la columna target_libre_final, renombrada
en la tabla de comparación como descripcion_referencia.

El modelo BLIP+LoRA libre base se entrenó directamente con target_libre_final.
El modelo BLIP+LoRA híbrido se entrenó con target_norm_blip_canonico, una
versión normalizada/canónica derivada del flujo libre previo, pero su evaluación
final también se realizó frente a target_libre_final para mantener una referencia
común entre todos los modelos.

El modelo con mejor rendimiento global fue {mejor_global['Modelo']}, con un
{mejor_global['% similitud aceptable']}% de salidas con similitud aceptable y
una similitud media del {mejor_global['similitud media']}%.
""".strip())
